<a href="https://colab.research.google.com/github/paulalcssantos/Pipeline-ETL-Women-in-Tech/blob/main/PipelineMulheresNaTecnologia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Desafio Pipeline ETL Integrado [WoMarkersCode]**



##**1. Configurando o Ambiente**

###1.1 - Instalação e Importação

In [ ]:
!pip install pandas prefect dbt-sqlite requests -q

In [ ]:
import pandas as pd
import requests
import json
import sqlite3

print("Ambiente configurado com sucesso!")

###1.2 - Fonte de Dados `CSV`

Arquivo do Kaggle apenas com mulheres (*perspectiva global*)

In [ ]:
!wget -O kaggle_survey_2022.csv "https://raw.githubusercontent.com/paulalcssantos/Desafio-Pipeline-WoMakersCode/refs/heads/main/kaggle_survey_2022_mulheres_dados.csv"

###1.3 - Fonte de Dados `SQL`

Criando e populando nosso banco de dados de origem (`SQL`)

In [ ]:
conn_bootcamp = sqlite3.connect("bootcampBI.db")
cursor_bootcamp = conn_bootcamp.cursor()

cursor_bootcamp.execute('''
CREATE TABLE IF NOT EXISTS PARTICIPANTES (
  ID_PARTICIPANTE INT,
  NOME VARCHAR(150),
  PAIS_ORIGEM VARCHAR(100)
)

''')

participantes = [
    (1, 'Maria', "Brazil"),
    (2, 'Luana', 'Portugal'),
    (3, 'Camila', 'Brazil'),
    (4, 'Luiza', 'Argentina'),
    (5, 'Silvia', 'Colombia'),
    (6, 'Paola', 'Brazil'),
    (7, 'Vitoria', 'Mexico'),
    (8, 'Caroline', 'Argentina'),
    (9, 'Marta', 'Portugal'),
    (10, 'Ana', 'Brazi')
]

cursor_bootcamp.executemany('INSERT INTO PARTICIPANTES VALUES (?,?,?)', participantes)

conn_bootcamp.commit()
conn_bootcamp.close()

print("Banco de Dados e tabela criados com sucesso!")

###1.4 - Fonte de Dados `JSON`

Dados semiestruturados

In [ ]:
%%writefile habilidades_categorias.json
{
  "Ferramentas de Análise": ["Python", "R", "SQL"],
  "Ferramentas de BI": ["Power Bi", "Tableau", "Looker"],
  "Plataformas de Nuvem": ["AWS", "Google Cloud", "Microsoft Azure"]
}

###1.4 - Data Warehouse

Criando o arquivo do nosso Data Warehouse (inicialmente vazio)

In [ ]:
conn_datawarehouse = sqlite3.connect('data_warehouse.db')
conn_datawarehouse.close()

print("Data Warehouse criado com sucesso!")




---






##**2. Extração e Carregamento - `CSV` e `SQL`**

###Três regras principais

* Modularidade
* Logging
* Tratamento de Erros

###2.1 - Configuração do Logger

In [ ]:
import logging

Registrando mensagens de nível INFO e acima em um arquivo chamado *'pipeline.log'* e criando um objeto logger para usar em nossas funções

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filename='pipeline.log',
    filemode='w'
)

logger = logging.getLogger()

print("Logging configurado com sucesso!")

###2.2 - Extração de dados de um Arquivo `CSV`

In [ ]:
def extrair_dados_kaggle(caminho_arquivo):
  try:
    logger.info(f"Inicando a extração do arquivo: {caminho_arquivo}")

    df = pd.read_csv(caminho_arquivo)

    logger.info(f"Extração do CSV concluída. {len(df)} linhas lidas e {len(df.columns)} colunas.")

    # Boa prática para verificar os tipos de dados que o Pandas encontrou
    logger.info(f"Tipos de dados das colunas: \n{df.dtypes}")

    return df

  except FileNotFoundError:
    logger.error(f"Arquivo não encontrado: {caminho_arquivo}")
    return None
  except Exception as e:
    logger.error(f"Ocorreu um erro inesperado ao extrair os dados: {e}")
    return None

#Teste da função
df_kaggle = extrair_dados_kaggle("kaggle_survey_2022.csv")

if df_kaggle is not None:
  display(df_kaggle.head())


###2.3 - Extração de Dados de um Banco `SQL`

In [ ]:
def extrair_dados_sql(caminho_banco):
  try:
    logger.info(f"Inicando a extração do banco de dados: {caminho_banco}")

    conexao = sqlite3.connect(caminho_banco)

    query = "SELECT * FROM PARTICIPANTES"

    df = pd.read_sql_query(query, conexao)

    logger.info(f"Extração do banco de dados concluída com sucesso!")
    return df

  except sqlite3.Error as e:
    logger.error(f"Ocorreu um erro ao conectar ao banco de dados")
    return None
  except Exception as e:
    logger.error(f"Ocorreu um erro inesperado ao extrair os dados: {e}")
    return None

  # O finally garante que a conexão com o banco seja sempre fechada, mesmo se der erro
  finally:
    if conexao:
      conexao.close()
      logger.info("Conexão com o banco de dados fechada.")

#Teste da função
df_participantes = extrair_dados_sql("bootcampBI.db")

if df_participantes is not None:
  display(df_participantes.head())

###2.4 - Função de Carregamento dos Dados

In [ ]:
def carregar_dados(df, nome_tabela, caminho_dw):
  if df is None:
    logger.warning("DataFrame vazio. Nada a ser carregado.")
    return

  try:
    logger.info(f"Iniciando o carregamento dos dados na tabela: {nome_tabela}")

    conexao_dw = sqlite3.connect(caminho_dw)

    df.to_sql(nome_tabela, conexao_dw, if_exists='replace', index=False)

    logger.info(f"Carga para a tabela '{nome_tabela}' concluída com sucesso!")

  except Exception as e:
      logger.error(f"Ocorreu um erro ao carregar os dados para a tabela '{nome_tabela}': {e}")

  finally:
    if conexao_dw:
      conexao_dw.close()
      logger.info("Conexão com o Data Warehouse fechada.")



Executa a carga dos DataFrames extraídos

In [ ]:
carregar_dados(df_kaggle, "kaggle_survey", "data_warehouse.db")
carregar_dados(df_participantes, "participantes", "data_warehouse.db")

##**3.Extração e Carregamento - `API` e `JSON`**

###3.1 -  Extração de Dados de uma API Web

In [ ]:
def extrair_dados_paises_api(url_api):
  try:
    logger.info(f"Iniciando a requisição à API: {url_api}")

    resposta = requests.get(url_api)

    if resposta.status_code == 200:
      dados_json = resposta.json()
      logger.info(f"Dados da API extraídos com sucessos. {len(dados_json)} registros de países.")
      return dados_json
    else:
      logger.error(f"Falha na requisição à API. Código de status: {resposta.status_code}")
      return None

  except Exception as e:
      logger.error(f"Ocorreu um erro inesperado ao extrair os dados")
      return None

url_paises = "https://restcountries.com/v3.1/all?fields=name,cca3,region"

#Teste da função
dados_paises = extrair_dados_paises_api(url_paises)

if dados_paises is not None:
  df_paises = pd.json_normalize(dados_paises)
  display(df_paises.head())

###3.2 - Extração de `JSON` (Simulando `NoSQL`) e Preparação para Transformação

In [ ]:
def extrair_categorias_habilidades_json(caminho_arquivo):
  try:
    logger.info(f"Iniciando a extração do arquivo JSON: {caminho_arquivo}")

    with open(caminho_arquivo, 'r') as arquivo:
      dados_json = json.load(arquivo)

    logger.info("Extração do JSON concluída com sucesso!")
    return dados_json

  except FileNotFoundError:
    logger.error(f"Arquivo não encontrado: {caminho_arquivo}")
    return None
  except Exception as e:
    logger.error(f"Ocorreu um erro inesperado ao extrair os dados: {e}")
    return None

#Teste da função
dados_habilidades = extrair_categorias_habilidades_json("habilidades_categorias.json")

if dados_habilidades is not None:
  df_habilidades = pd.json_normalize(dados_habilidades)
  display(df_habilidades.head())

###3.3 - Tabulação do `JSON`

In [ ]:
def transformar_json_em_df(dados_json):
  if dados_json is None:
    logger.warning("Dados JSON vazios. Nada a ser transformado")
    return None

  lista_categorias = []

  for categoria, habilidades in dados_json.items():
    for habilidade in habilidades:
      lista_categorias.append({"CATEGORIA": categoria, "HABILIDADE" : habilidade})

  df_categorias = pd.DataFrame(lista_categorias)

  logger.info("Transformação do JSON em DataFrame concluída com sucesso!")
  return df_categorias

#Teste da função
df_categorias = transformar_json_em_df(dados_habilidades)

if df_categorias is not None:
  display(df_categorias.head())

###3.5 - Carregamento dos DataFrames

In [ ]:
carregar_dados(df_paises, "paises", "data_warehouse.db")
carregar_dados(df_categorias, "categorias", "data_warehouse.db")

##**4. Transformação com dbt**

###4.1 Criação do projeto dbt

O dbt init é como o 'Arquivo > Novo Projeto'. Ele cria toda a estrutura de pastas que um projeto dbt precisa.

In [ ]:
!dbt init pipeline_mulheres_na_tecnologia

###4.2 Configuração do arquivo profiles.yml

* Importação da biblioteca `yaml`, que nos permite criar estruturas Python (como dicionários) e convertê-las facilmente para o formato de texto `YAML`, o que é muito mais seguro do que tentar escrever o texto manualmente.
* Importação da biblioteca `os`, que permite interagir com o sistema operacional. Precisamos dela para encontrar o diretório "home" do usuário (`~` ou `/root/`) e garantir que a pasta `.dbt` exista antes de tentarmos salvar nosso arquivo lá.

In [ ]:
import yaml
import os

O dbt cria o profiles.yml em um diretório oculto e aqui vamos sobrescrevê-lo.

In [ ]:
profiles_config = {
    'pipeline_mulheres_na_tecnologia': {
        'target': 'dev',
        'outputs': {
            'dev': {
                'type': 'sqlite',
                'threads': 1,

                # Parâmetro para versões mais recentes
                'schemas_and_paths': {
                    'main': '../data_warehouse.db'
                },

                # Parâmetro para versões mais antigas
                'database': '../data_warehouse.db',

                # Parâmetro para versões mais antigas
                'schema': 'main',

                # Parâmetro opcional, mas boa prática
                'schema_directory': '.'
            }
        }
    }
}

# Caminho onde o dbt espera encontrar o arquivo de perfis
dbt_profile_dir = os.path.expanduser('~/.dbt/')
os.makedirs(dbt_profile_dir, exist_ok=True)
profiles_path = os.path.join(dbt_profile_dir, 'profiles.yml')

with open(profiles_path, 'w') as f:
    yaml.dump(profiles_config, f)

###4.3 Testando a conexão

In [ ]:
%cd pipeline_mulheres_na_tecnologia
!dbt debug

##**5. Criando Modelos de *Staging* com dbt**

###5.1 Configurando as Fontes

Antes de escrever nosso SQL, precisamos dizer ao dbt onde encontrar nossos dados brutos. Fazemos isso em um arquivo de configuração `.yml `dentro da pasta models. Essa é uma das melhores práticas do dbt, pois nos permite criar um 'dicionário de dados' e testar a qualidade das nossas fontes.

In [ ]:
%%writefile /content/pipeline_mulheres_na_tecnologia/models/staging/source.yml
version: 2

sources:
  - name: dados_brutos
    database: data_warehouse.db
    schema: main

    tables:
      - name: kaggle_survey
        description: "Dados brutos da pesquisa Kaggle, filtrado por mulheres na área de dados"
      - name: participantes
        description: "Dados das participantes do bootcamp"
      - name: paises
        description: "Dados dos países extraídos da API REST Countries"
      - name: categorias
        description: "Mapeamento das habilidades e categorias, extraído de um arquivo JSON"

###5.2 Criando o Primeiro Modelo de Staging

In [ ]:
%%writefile /content/pipeline_mulheres_na_tecnologia/models/staging/stg_kaggle_survey.sql

SELECT PAIS,
        NIVEL_EDUCACIONAL,
        ANOS_PROGRAMANDO,
        CARGO_ATUAL,
        ANOS_USANDO_ML,
        CASE WHEN SALARIO_ANUAL_USD LIKE '%-%'
             THEN CAST(REPLACE(SUBSTR(SALARIO_ANUAL_USD, 1, INSTR(SALARIO_ANUAL_USD, '-') - 1), ',', '') AS REAL)
             WHEN SALARIO_ANUAL_USD IS NOT NULL
             THEN CAST(REPLACE(SALARIO_ANUAL_USD, ',', '') AS REAL)
             ELSE NULL
        END AS SALARIO_ANUAL_USD,
        LINGUAGENS_USADAS,
        BANCOS_DE_DADOS_USADOS,
        FERRAMENTAS_BI_USADAS
FROM {{ source('dados_brutos', 'kaggle_survey') }}

In [ ]:
!dbt run

##**6. Criando Modelos Finais *(Marts)* com dbt**

###6.1 - Staging para Participantes

In [ ]:
%%writefile /content/pipeline_mulheres_na_tecnologia/models/staging/stg_participantes.sql
SELECT ID_PARTICIPANTE,
       NOME,
       PAIS_ORIGEM
FROM {{ source('dados_brutos', 'participantes') }}

###6.2 - Staging para Países

In [ ]:
%%writefile /content/pipeline_mulheres_na_tecnologia/models/staging/stg_paises.sql
SELECT "name.common" AS NOME_PAIS,
       cca3          AS CODIGO_PAIS,
       region        AS REGIAO
FROM {{ source('dados_brutos', 'paises') }}

###6.3 - Criando o modelo final (marts) - dim_desenvolvedoras

Este modelo une os dados do survey do Kaggle com os dados de países da API.

Ele representa nossa tabela de dimensão final sobre as desenvolvedoras.

In [ ]:
%%writefile /content/pipeline_mulheres_na_tecnologia/models/marts/dim_desenvolvedoras.sql

-- Configuração de materialização: vamos criar como uma tabela física.
{{config(materialized = 'table')}}

WITH stg_kaggle AS (
    SELECT * FROM {{ref('stg_kaggle_survey')}}
),
stg_paises AS (
    SELECT * FROM {{ref('stg_paises')}}
)

SELECT stg_kaggle.PAIS,
       stg_paises.CODIGO_PAIS,
       stg_paises.REGIAO,
       stg_kaggle.NIVEL_EDUCACIONAL,
       stg_kaggle.ANOS_PROGRAMANDO,
       stg_kaggle.CARGO_ATUAL,
       stg_kaggle.ANOS_USANDO_ML,
       stg_kaggle.SALARIO_ANUAL_USD,
       stg_kaggle.LINGUAGENS_USADAS,
       stg_kaggle.BANCOS_DE_DADOS_USADOS,
       stg_kaggle.FERRAMENTAS_BI_USADAS
FROM stg_kaggle
   LEFT JOIN stg_paises ON stg_kaggle.PAIS = stg_paises.NOME_PAIS

###6.4 - Executando o dbt

In [ ]:
!dbt run

##**7. Orquestração e Monitoramento do Pipeline com Prefect**

In [ ]:
# ==============================================================================
# CÉLULA COMPLETA: DEFINIÇÃO E EXECUÇÃO DO PIPELINE ORQUESTRADO
# ==============================================================================

# 1. Instalações
!pip install pandas prefect dbt-sqlite requests -q

# 2. Imports
import pandas as pd
import requests
import json
import sqlite3
from prefect import task, flow, get_run_logger
import subprocess
import os
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filename='pipeline.log',
    filemode='w'
)

logger = logging.getLogger()

print("Logging configurado com sucesso!")

# --- TAREFAS DE EXTRAÇÃO E CARGA (EL) ---
# Cada função de extração e carga que já construímos agora é uma @task do Prefect.

@task(retries=3, retry_delay_seconds=5)
def extrair_dados_kaggle(caminho_arquivo):
  try:
    logger.info(f"Inicando a extração do arquivo: {caminho_arquivo}")

    df = pd.read_csv(caminho_arquivo)

    logger.info(f"Extração do CSV concluída. {len(df)} linhas lidas e {len(df.columns)} colunas.")

    logger.info(f"Tipos de dados das colunas: \n{df.dtypes}")

    return df

  except FileNotFoundError:
    logger.error(f"Arquivo não encontrado: {caminho_arquivo}")
    return None
  except Exception as e:
    logger.error(f"Ocorreu um erro inesperado ao extrair os dados: {e}")
    return None

@task
def extrair_dados_sql(caminho_banco):
  try:
    logger.info(f"Inicando a extração do banco de dados: {caminho_banco}")

    conexao = sqlite3.connect(caminho_banco)

    query = "SELECT * FROM PARTICIPANTES"

    df = pd.read_sql_query(query, conexao)

    logger.info(f"Extração do banco de dados concluída com sucesso!")
    return df

  except sqlite3.Error as e:
    logger.error(f"Ocorreu um erro ao conectar ao banco de dados")
    return None
  except Exception as e:
    logger.error(f"Ocorreu um erro inesperado ao extrair os dados: {e}")
    return None

  finally:
    if conexao:
      conexao.close()
      logger.info("Conexão com o banco de dados fechada.")

@task
def extrair_categorias_habilidades_json(caminho_arquivo):
  try:
    logger.info(f"Iniciando a extração do arquivo JSON: {caminho_arquivo}")

    with open(caminho_arquivo, 'r') as arquivo:
      dados_json = json.load(arquivo)

    logger.info("Extração do JSON concluída com sucesso!")

    if dados_json is None:
      logger.warning("Dados JSON vazios. Nada a ser transformado")
    return None

    lista_categorias = []

    for categoria, habilidades in dados_json.items():
      for habilidade in habilidades:
        lista_categorias.append({"CATEGORIA": categoria, "HABILIDADE" : habilidade})

    df_categorias = pd.DataFrame(lista_categorias)

    logger.info("Transformação do JSON em DataFrame concluída com sucesso!")
    return df_categorias

  except FileNotFoundError:
    logger.error(f"Arquivo não encontrado: {caminho_arquivo}")
    return None
  except Exception as e:
    logger.error(f"Ocorreu um erro inesperado ao extrair os dados: {e}")
    return None

@task
def extrair_dados_paises_api(url_api):
  try:
    logger.info(f"Iniciando a requisição à API: {url_api}")

    resposta = requests.get(url_api)

    if resposta.status_code == 200:
      dados_json = resposta.json()
      logger.info(f"Dados da API extraídos com sucessos. {len(dados_json)} registros de países.")
      return dados_json
    else:
      logger.error(f"Falha na requisição à API. Código de status: {resposta.status_code}")
      return None

  except Exception as e:
      logger.error(f"Ocorreu um erro inesperado ao extrair os dados")
      return None

@task
def carregar_dados(df, nome_tabela, caminho_dw):
  if df is None:
    logger.warning("DataFrame vazio. Nada a ser carregado.")
    return

  try:
    logger.info(f"Iniciando o carregamento dos dados na tabela: {nome_tabela}")

    conexao_dw = sqlite3.connect(caminho_dw)

    df.to_sql(nome_tabela, conexao_dw, if_exists='replace', index=False)

    logger.info(f"Carga para a tabela '{nome_tabela}' concluída com sucesso!")

  except Exception as e:
      logger.error(f"Ocorreu um erro ao carregar os dados para a tabela '{nome_tabela}': {e}")

  finally:
    if conexao_dw:
      conexao_dw.close()
      logger.info("Conexão com o Data Warehouse fechada.")


# --- TAREFAS DE TRANSFORMAÇÃO (T) ---
# Tarefas que executam os comandos do dbt via terminal.

@task
def executar_dbt_run():
    logger = get_run_logger()
    logger.info("Iniciando 'dbt run'...")
    try:
        subprocess.run(['dbt', 'run'], check=True, cwd='/content/pipeline_mulheres_na_tecnologia')
        logger.info("'dbt run' concluído com sucesso.")
    except subprocess.CalledProcessError as e:
        logger.error(f"Falha no 'dbt run': {e}")
        raise

# --- O FLUXO PRINCIPAL (@flow) ---
# Orquestra a execução de todas as tarefas na ordem correta.

@flow(name="Pipeline ETL - Mulheres na Tecnologia")
def pipeline_principal():
    logger = get_run_logger()
    logger.info("### INICIANDO O PIPELINE DE ETL ###")

    # --- Fase 1: Extração e Carga (EL) ---
    # O Prefect executa estas tarefas em paralelo
    df_kaggle = extrair_dados_kaggle('/content/kaggle_survey_2022.csv')
    df_participantes = extrair_dados_sql('/content/bootcampBI.db')
    df_paises = pd.json_normalize(extrair_dados_paises_api("https://restcountries.com/v3.1/all?fields=name,cca3,region"))
    dados_habilidades = extrair_categorias_habilidades_json("/content/habilidades_categorias.json")

    # As cargas dependem das extrações (dependência implícita)
    carga_kaggle = carregar_dados(df_kaggle, 'kaggle_survey', 'data_warehouse.db')
    carga_participantes = carregar_dados(df_participantes, 'participantes', 'data_warehouse.db')
    carga_paises = carregar_dados(df_paises, 'paises', 'data_warehouse.db')
    carga_habilidades = carregar_dados(dados_habilidades, 'categorias', 'data_warehouse.db')

    # --- Fase 2: Transformação (T) ---
    # A transformação com dbt só pode começar depois que TODAS as cargas terminarem.
    # Usamos 'wait_for' para criar essa dependência explícita.
    dbt_run = executar_dbt_run(wait_for=[carga_kaggle, carga_participantes, carga_paises, carga_habilidades])

    logger.info("### PIPELINE DE ETL CONCLUÍDO COM SUCESSO ###")

# --- EXECUÇÃO DO FLUXO ---
if __name__ == "__main__":
    pipeline_principal()